# Unsloth Deep Dive #00 — 2026 Migration Notes

If you're writing new code from older blogs/Colabs, here's what changed.

**Note:** This notebook is built around the **official 2026 Unsloth notebooks**:
- `Qwen3_(4B)_Instruct.ipynb`, `Qwen3_(4B)_Thinking.ipynb`
- `Gemma4_(E2B)_Text.ipynb`
- `Qwen3_5_(2B)_Vision.ipynb`

---

## The Big Picture — Five Major Changes

| # | Change | Impact |
|---|--------|--------|
| 1 | **`unsloth.chat_templates` module** | `get_chat_template`, `standardize_data_formats`, `train_on_responses_only` are now the official pattern |
| 2 | **`FastModel` (new unified API)** | Preferred for Gemma 4 and VLMs; `FastLanguageModel` is still the recommended choice for Qwen3 |
| 3 | **`FastVisionModel` is required (vision SFT)** | Used together with `UnslothVisionDataCollator` |
| 4 | **Dynamic 2.0 quantization** | `unsloth-bnb-4bit` suffix exists for some models (not all) |
| 5 | **New model variants** | Qwen3-4B-Instruct-2507, Thinking-2507; Gemma 4 E/26B-A4B; Qwen3.5 multimodal |

## Change 1 — The `unsloth.chat_templates` Module

**The most critical change.** Most older guides recommended `assistant_only_loss=True` (TRL native). The official Unsloth pattern is different:

```python
from unsloth.chat_templates import (
    get_chat_template,           # Apply a chat template to the tokenizer
    standardize_data_formats,    # Normalize the dataset format
    train_on_responses_only,     # Loss masking — Unsloth's approach
)
```

### Old (TRL native)
```python
trainer = SFTTrainer(
    ...
    args = SFTConfig(
        assistant_only_loss = True,        # TRL's approach
    ),
)
```

**Problems:**
- Requires the chat template to contain the `{% generation %}` keyword
- Breaks on VLMs (`_is_vlm` check)
- On some models you end up with all labels = -100

### New (Unsloth official)
```python
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(...)  # build the trainer normally first

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
```

**Advantages:**
- Manual token specification — no dependency on the chat template
- Independent of the VLM check
- Works on every model

### Tokens Are Model-Specific

| Model | instruction_part | response_part |
|-------|------------------|---------------|
| Qwen3 (Instruct/Thinking) | `<\|im_start\|>user\n` | `<\|im_start\|>assistant\n` |
| Gemma 4 | `<\|turn>user\n` | `<\|turn>model\n` |
| Llama 3 | `<\|start_header_id\|>user<\|end_header_id\|>\n\n` | `<\|start_header_id\|>assistant<\|end_header_id\|>\n\n` |

## Change 2 — `FastModel` (New Unified API)

**Important correction:** Contrary to my previous claim, `FastLanguageModel` is NOT legacy — the official Qwen3 4B Instruct and Thinking notebooks use it.

### Hierarchy (verified on the server)

```
FastBaseModel
  ├── FastModel              ← NEW unified API (Gemma 4, VLM text-only)
  │   ├── FastTextModel      ← FastModel's text version
  │   └── FastVisionModel    ← For vision SFT
  └── FastLlamaModel
      └── FastLanguageModel  ← For Qwen3, Llama, Mistral (STILL recommended)
```

### When To Use Which?

| Scenario | Class used in the official notebook |
|----------|-------------------------------------|
| Qwen3 (Instruct/Thinking) text SFT | **`FastLanguageModel`** |
| Gemma 4 E2B text SFT | **`FastModel`** |
| Qwen3.5 Vision SFT | **`FastVisionModel`** |
| Llama 3 / Mistral text SFT | `FastLanguageModel` (general consensus) |

Both are actively used — there is no single "new correct answer".

## Change 3 — `UnslothVisionDataCollator` for Vision SFT

For vision SFT a **plain `SFTTrainer` does not work**. You need extra config + a collator:

```python
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),  # REQUIRED
    train_dataset = converted_dataset,
    args = SFTConfig(
        # Required trio for vision:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
        ...
    ),
)
```

**Important:** In vision SFT do NOT use `train_on_responses_only` — the collator handles masking itself.

## Change 4 — Dynamic 2.0 Quantization

There are two kinds of 4-bit variant on the Hub:

- `model-bnb-4bit` — standard bitsandbytes
- `model-unsloth-bnb-4bit` — Dynamic 2.0 (outlier-aware, ~2-5% better)

### Important: Not Available for Every Model

```python
# Check whether it exists
from huggingface_hub import list_models
for m in list_models(author="unsloth", search="YOUR_MODEL"):
    print(m.id)
```

**Current state (May 2026):**
- Qwen3 (text) → `unsloth-bnb-4bit` AVAILABLE
- Gemma 4 E2B/E4B → `unsloth-bnb-4bit` AVAILABLE
- **Qwen3.5 (VLM) → ONLY full + GGUF, NO bnb-4bit!**

For Qwen3.5, `load_in_4bit=True` does runtime quantization that gets the job done, but you don't get the Dynamic 2.0 advantage.

### In the Official Notebooks

Interesting finding: most official notebooks **start from the full-precision model** (`unsloth/Qwen3-4B-Instruct-2507`) and only list the pre-quantized variants as comments. So Dynamic 2.0 is an option, not the default.

## Change 5 — New Model Variants

### Qwen3 — `2507` versions
- `unsloth/Qwen3-4B-Instruct-2507`
- `unsloth/Qwen3-4B-Thinking-2507`
- Updated releases of the older Qwen3 models

### Gemma 4 — New naming
- `unsloth/gemma-4-E2B`, `gemma-4-E4B` — Elastic (multimodal capable)
- `unsloth/gemma-4-26B-A4B` — MoE (26B total, 4B active)
- `unsloth/gemma-4-31B` — Dense

### Qwen3.5 — Multimodal Family
- 0.8B, 2B, 4B, 9B, 27B (dense)
- 35B-A3B, 122B-A10B (MoE)
- All `Qwen3_5ForConditionalGeneration` (VLM architecture)

### Thinking Models Are Special — `enable_thinking`

```python
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,        # NEW — toggle reasoning mode
)
```

In training, all data should contain the thinking blocks; at inference you can optionally turn them off.

## Changed Default Values

Values used by the official notebooks that I had wrong in my earlier guide:

| Param | Wrong (my old guide) | Correct (official notebook) |
|-------|----------------------|------------------------------|
| `weight_decay` | 0.01 | **0.001** |
| `lr_scheduler_type` | `"cosine"` | **`"linear"`** |
| `warmup` | `warmup_ratio=0.03` | **`warmup_steps=5`** |
| `random_state` | 42 | **3407** (Unsloth's standard seed) |
| `processing_class=` | (TRL 1.x) | **`tokenizer=`** (in the official notebooks) |

### `[NEW!]` Flags (left as comments in the official notebooks)

- `load_in_8bit = False` — "A bit more accurate, uses 2x memory"
- `full_finetuning = False` — "We have full finetuning now!"
- `use_gradient_checkpointing = "unsloth"` — "30% less VRAM, fits 2x larger batch"


## Migration Checklist

### 1. Import `unsloth.chat_templates`
```python
from unsloth.chat_templates import (
    get_chat_template, standardize_data_formats, train_on_responses_only,
)
```

### 2. Apply the chat template
```python
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
```

### 3. Normalize the dataset
```python
dataset = standardize_data_formats(dataset)
```

### 4. Build the text field with a formatting function
```python
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    return {"text": [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]}
dataset = dataset.map(formatting_prompts_func, batched=True)
```

### 5. Wrap the trainer with `train_on_responses_only`
```python
trainer = SFTTrainer(...)  # do NOT use `assistant_only_loss=True`
trainer = train_on_responses_only(trainer, instruction_part=..., response_part=...)
```

### 6. Switch to the official defaults
```python
SFTConfig(
    ...,
    weight_decay = 0.001,
    lr_scheduler_type = "linear",
    warmup_steps = 5,
    seed = 3407,
)
```

### 7. Don't forget `add_generation_prompt=True` at inference
```python
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

## Summary

The minimum you need to know from the official Unsloth notebooks:

✅ The `unsloth.chat_templates` module — `get_chat_template`, `standardize_data_formats`, `train_on_responses_only`
✅ No more `assistant_only_loss=True` — use `train_on_responses_only(trainer, instruction_part, response_part)`
✅ `FastLanguageModel` (Qwen3), `FastModel` (Gemma 4), `FastVisionModel` (Vision) — pick by context
✅ For vision SFT: `UnslothVisionDataCollator` + the required config trio
✅ `weight_decay=0.001`, `lr_scheduler_type="linear"`, `warmup_steps=5`, `seed=3407`
✅ Tokens are model-specific: Qwen3 `<|im_start|>`, Gemma `<|turn>`, Llama `<|start_header_id|>`

**Next notebook:** `01_internals.ipynb` — Triton kernels, patching, internals.